<a href="https://colab.research.google.com/github/kayorde25/Plant-Seedling-Classification-using-EfficientNetB0/blob/main/Plant_seedling_classification_EfficientNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Project Title

Plant Seedling Classification using EfficientNetB0

## Problem Statement
Plant seedling classification is an important computer vision task in agriculture and precision farming. The goal of this project is to classify plant seedlings into their correct species using deep learning. This study applies EfficientNetB0 through transfer learning to build an accurate and efficient image classification model.

####Import Libraries

In [ ]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


####Load and Inspect Dataset
In this section, the dataset is loaded from class-specific folders and organized into a dataframe containing image paths and labels.

In [ ]:
# ============================================
# 2. SET PARAMETERS
# ============================================
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = None   # will be set automatically
MODEL_PATH = "best_efficientnet_model.keras"

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "data"

images_path = os.path.join(DATA_DIR, "images.npy")
labels_path = os.path.join(DATA_DIR, "Labels.csv")

# Check dataset exists
if not os.path.exists(images_path) or not os.path.exists(labels_path):
    raise FileNotFoundError(
        "Dataset not found.\n"
        "Download from: https://www.kaggle.com/datasets/abiolaoolaleye/computer-vision\n"
        "Place files in the 'data/' directory."
    )

# Load data
images = np.load(images_path)
labels_df = pd.read_csv(labels_path)

print("Images shape:", images.shape)
print("Labels shape:", labels_df.shape)

FileNotFoundError: Dataset not found.
Download from: https://www.kaggle.com/datasets/abiolaoolaleye/computer-vision
Place files in the 'data/' directory.

### 1. Install the Kaggle API client

In [3]:
pip install kaggle

In [ ]:
# ============================================
# 1. UPLOAD kaggle.json
# ============================================
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

### 2. Configure Kaggle API Key

To allow Colab to access your Kaggle account, you need to provide your Kaggle API credentials. Follow these steps:

1.  **Download your `kaggle.json` file:**
    *   Go to your Kaggle account page: [https://www.kaggle.com/me/account](https://www.kaggle.com/me/account)
    *   Scroll down to the 'API' section.
    *   Click 'Create New API Token' to download the `kaggle.json` file.
    *   Open `kaggle.json` with a text editor. It will contain your username and key, like this:
        ```json
        {"username":"abiolaoolaleye","key":"GAT_246c031a36d4ece59487e86fed12fd4a"}
        ```

2.  **Add your credentials to Colab Secrets:**
    *   In Google Colab, look for the '🔑' icon (Secrets) in the left sidebar and click on it.
    *   Click 'Add new secret'.
    *   For the 'Name' field, type `KAGGLE_USERNAME` and paste your Kaggle username (from `kaggle.json`) into the 'Value' field.
    *   Click 'Add new secret' again.
    *   For the 'Name' field, type `KAGGLE_KEY` and paste your Kaggle API key (from `kaggle.json`) into the 'Value' field.

After adding the secrets, run the cell below to set up the environment variables and create the `kaggle.json` file in the correct location.

### 3. Download and Extract the Dataset

Now that your Kaggle API key is configured, we can download the dataset from Kaggle and extract its contents into the `data/` directory.

In [ ]:
import os
import zipfile

DATA_DIR = "data"
DATASET_SLUG = "abiolaoolaleye/computer-vision"

# Create data directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

# Download the dataset
print(f"Downloading dataset {DATASET_SLUG}...")
!kaggle datasets download -d {DATASET_SLUG} -p {DATA_DIR}

# Find the downloaded zip file (assuming it's named after the slug or dataset title)
# Kaggle usually names the zip file after the dataset title, which is 'computer-vision.zip' for this slug
zip_file_path = os.path.join(DATA_DIR, 'computer-vision.zip')

if os.path.exists(zip_file_path):
    print(f"Extracting {zip_file_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    os.remove(zip_file_path) # Clean up the zip file
    print("Extraction complete. Zip file removed.")
else:
    print(f"Error: Zip file not found at {zip_file_path}. Please check Kaggle download or dataset slug.")

# Verify files are in data directory
print("Files in data directory after extraction:")
print(os.listdir(DATA_DIR))

### 4. Re-run the data loading cell

Now that the dataset files (`images.npy` and `Labels.csv`) should be in the `data/` directory, please re-run the cell `zfXvCYFu11OJ` to load them into your notebook.

In [4]:
from google.colab import userdata
import os

# Set environment variables from Colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Create the .kaggle directory if it doesn't exist
KAGGLE_DIR = os.path.expanduser('~/.kaggle')
os.makedirs(KAGGLE_DIR, exist_ok=True)

# Create kaggle.json file with the API credentials
kaggle_json_path = os.path.join(KAGGLE_DIR, 'kaggle.json')
with open(kaggle_json_path, 'w') as f:
    f.write(f'{{"username":"{os.environ["KAGGLE_USERNAME"]}","key":"{os.environ["KAGGLE_KEY"]}"}}')

# Set permissions for kaggle.json
os.chmod(kaggle_json_path, 0o600)

print("Kaggle API key configured successfully.")

SecretNotFoundError: Secret KAGGLE_USERNAME does not exist.

### Important: Download the Dataset

To continue with the project, you *must* download the actual `images.npy` and `Labels.csv` files from the Kaggle dataset:

**Dataset Link:** https://www.kaggle.com/datasets/abiolaoolaleye/computer-vision

After downloading, place these two files directly into the `data/` directory that was created by the previous code cell. You will need to replace the dummy files if they were created.


#### Encode Labels
The class labels are converted into numeric form so they can be used in model training.

## Train, Validation, and Test Split
Split Dataset
The dataset is divided into training, validation, and test sets to support model training and unbiased evaluation.

#### Image Preprocessing
Images are resized to 224 × 224 pixels and preprocessed using EfficientNet-specific preprocessing. Labels are converted to one-hot encoded vectors.

#### Data Augmentation
Data augmentation is applied to improve generalization by exposing the model to slightly modified versions of the training images.

#### Create TensorFlow Datasets
Build TensorFlow Datasets: Efficient and scalable tf.data pipelines are created for training, validation, and testing.

#### Visualize Samples
Visualize Sample Images: A few example images are displayed to confirm that the dataset has loaded correctly.

#### Build EfficientNetB0 Model
Build the EfficientNetB0 Model: EfficientNetB0 pretrained on ImageNet is used as the base model. A custom classification head is added for the plant seedling classes.

#### Train Stage 1
Feature Extraction: At this stage, the EfficientNet base is frozen and only the custom classification head is trained.

#### Plot Training Curves
Training Curves: Training and validation accuracy and loss are plotted to monitor learning progress and possible overfitting.

#### Fine-Tuning
#####Stage 2: Fine-Tuning - The upper layers of EfficientNetB0 are unfrozen and trained with a smaller learning rate so the model can better adapt to the plant seedling dataset.

#### Final Evaluation
Final Evaluation: The best saved model is evaluated on the test set using accuracy, classification report, and confusion matrix.
16. Conclusion

## Conclusion
This project demonstrated how EfficientNetB0 can be used effectively for plant seedling classification through transfer learning and fine-tuning. The final model achieved strong performance and shows how modern pretrained CNNs can be adapted for practical image classification tasks.